<a href="https://colab.research.google.com/github/hub-naveen/Road_Potholes_Detection/blob/main/Potholes_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import sys
import subprocess

def install_dependencies():
    print("[1/15] Installing dependencies...")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "ultralytics", "kaggle", "roboflow", "matplotlib", "seaborn", "onnx", "tensorflow"])
        print("Installation successful.")
    except Exception as e:
        print(f"Installation failed: {e}")

install_dependencies()

[1/15] Installing dependencies...
Installation successful.


In [ ]:
import torch
from ultralytics import YOLO
import yaml
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import cv2
from google.colab import files
from pathlib import Path

def check_gpu():
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"[2/15] Using device: {device}")
    return device

DEVICE = check_gpu()

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
[2/15] Using device: cpu


In [83]:
import os
from google.colab import files

def authenticate_kaggle():
    print("[3/15] Authenticating Kaggle...")
    kaggle_dir = os.path.expanduser('~/.kaggle')
    os.makedirs(kaggle_dir, exist_ok=True)

    if not os.path.exists(os.path.join(kaggle_dir, 'kaggle.json')):
        print("Please upload your kaggle.json file (from Kaggle Account settings).")
        uploaded = files.upload()
        for fn in uploaded.keys():
            if fn == 'kaggle.json':
                with open(os.path.join(kaggle_dir, 'kaggle.json'), 'wb') as f:
                    f.write(uploaded[fn])
                os.chmod(os.path.join(kaggle_dir, 'kaggle.json'), 0o600)
                print("Kaggle API token configured.")
    else:
        print("Kaggle API token already exists.")

authenticate_kaggle()

[3/15] Authenticating Kaggle...
Please upload your kaggle.json file (from Kaggle Account settings).


In [85]:
import shutil
import os

def download_kaggle_dataset():
    print("[4/15] Downloading Pothole Dataset from Kaggle...")
    dataset_slug = "chitholian/pothole-detection-dataset"
    data_path = "/content/dataset"
    kaggle_config = os.path.expanduser('~/.kaggle/kaggle.json')

    if not os.path.exists(kaggle_config):
        print("Error: kaggle.json not found. Please run the previous cell to upload it.")
        return

    try:
        # Import Kaggle inside the function to avoid early initialization errors
        from kaggle.api.kaggle_api_extended import KaggleApi

        if os.path.exists(data_path):
            shutil.rmtree(data_path)
        os.makedirs(data_path, exist_ok=True)

        api = KaggleApi()
        api.authenticate()

        print(f"Downloading {dataset_slug}...")
        api.dataset_download_files(dataset_slug, path=data_path, unzip=True)
        print(f"Dataset successfully downloaded and extracted to {data_path}")
        print("Directory structure:", os.listdir(data_path))
    except Exception as e:
        print(f"Dataset download failed: {e}")

download_kaggle_dataset()

[4/15] Downloading Pothole Dataset from Kaggle...
Error: kaggle.json not found. Please run the previous cell to upload it.


In [86]:
import glob
import matplotlib.image as mpimg

def explore_dataset():
    print("[5/15] Exploring Dataset...")
    data_path = "/content/dataset"
    img_exts = ['*.jpg', '*.jpeg', '*.png']

    images = []
    for ext in img_exts:
        images.extend(glob.glob(f"{data_path}/**/{ext}", recursive=True))

    print(f"Total images found: {len(images)}")

    if len(images) > 0:
        plt.figure(figsize=(12, 8))
        for i in range(min(4, len(images))):
            plt.subplot(2, 2, i+1)
            img = mpimg.imread(images[i])
            plt.imshow(img)
            plt.axis('off')
            plt.title(f"Sample {i+1}")
        plt.tight_layout()
        plt.show()
    else:
        print("No images found for exploration. Check dataset path.")

explore_dataset()

[5/15] Exploring Dataset...
Total images found: 0
No images found for exploration. Check dataset path.


In [87]:
def create_yaml_config():
    print("[6/15] Creating YOLOv8 YAML Configuration...")
    data_path = "/content/dataset"

    # Dynamic path detection for train/val based on standard Kaggle structures
    config = {
        'path': data_path,
        'train': 'train/images',
        'val': 'valid/images',
        'test': 'test/images',
        'names': {0: 'pothole'}
    }

    yaml_path = '/content/pothole_config.yaml'
    with open(yaml_path, 'w') as f:
        yaml.dump(config, f, default_flow_style=False)

    print(f"Configuration saved to {yaml_path}")
    return yaml_path

YAML_PATH = create_yaml_config()

[6/15] Creating YOLOv8 YAML Configuration...
Configuration saved to /content/pothole_config.yaml


In [88]:
def train_model(yaml_path):
    print("[7/15] Initializing YOLOv8 Training...")
    model = YOLO('yolov8n.pt')

    results = model.train(
        data=yaml_path,
        epochs=25,
        imgsz=640,
        batch=16,
        lr0=0.01,
        device=DEVICE,
        name='pothole_detection',
        exist_ok=True
    )
    print("Training completed successfully.")
    return model

model = train_model(YAML_PATH)

[7/15] Initializing YOLOv8 Training...
Ultralytics 8.4.67 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/pothole_config.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=25, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=pothole_detection, nbs=64, nms=False, opset=None, optimize=False,

RuntimeError: Dataset '/content/pothole_config.yaml' error ❌ Dataset '/content/pothole_config.yaml' images not found, missing path '/content/dataset/valid/images'
Note dataset download directory is '/content/datasets'. You can update this in '/root/.config/Ultralytics/settings.json'

In [101]:
def evaluate_model(model):
    print("[8/15] Evaluating Model Performance...")
    try:
        metrics = model.val()
        print(f"mAP@50: {metrics.box.map50:.4f}")
        print(f"mAP@50-95: {metrics.box.map:.4f}")
        print(f"Precision: {metrics.box.mp:.4f}")
        print(f"Recall: {metrics.box.mr:.4f}")
        return metrics
    except Exception as e:
        print(f"Evaluation failed: {e}. Ensure training completed successfully.")

evaluate_model(model)

[8/15] Evaluating Model Performance...
Evaluation failed: 'NoneType' object has no attribute 'val'. Ensure training completed successfully.


In [102]:
def visualize_results():
    print("[9/15] Generating Training Visualizations...")
    results_dir = Path('/content/runs/detect/pothole_detection')

    if results_dir.exists():
        if (results_dir / 'results.png').exists():
            plt.figure(figsize=(15, 10))
            img = mpimg.imread(str(results_dir / 'results.png'))
            plt.imshow(img)
            plt.axis('off')
            plt.title("Training Metrics and Loss Curves")
            plt.show()

        if (results_dir / 'confusion_matrix.png').exists():
            plt.figure(figsize=(10, 10))
            img = mpimg.imread(str(results_dir / 'confusion_matrix.png'))
            plt.imshow(img)
            plt.axis('off')
            plt.title("Confusion Matrix")
            plt.show()
    else:
        print("Results directory not found. Visualizations skipped.")

visualize_results()

[9/15] Generating Training Visualizations...


In [103]:
def export_model(model):
    print("[10/15] Exporting Model to ONNX and TFLite...")
    try:
        onnx_path = model.export(format='onnx')
        tflite_path = model.export(format='tflite')
        print(f"Model exported to: {onnx_path} and {tflite_path}")
    except Exception as e:
        print(f"Export failed: {e}")

export_model(model)

[10/15] Exporting Model to ONNX and TFLite...
Export failed: 'NoneType' object has no attribute 'export'


In [104]:
def run_custom_inference():
    print("[11/15] Custom Inference Module...")
    test_images = list(Path('/content/dataset/test/images').glob('*.jpg'))
    if test_images:
        results = model.predict(source=str(test_images[0]), save=True, conf=0.25)
        print(f"Inference complete. Results saved to {results[0].save_dir}")
    else:
        print("No test images found for sample inference.")

run_custom_inference()

[11/15] Custom Inference Module...
No test images found for sample inference.


In [110]:
def deployment_example():
    print("[12/15] Production Deployment Example (FastAPI/Flask Structure)")
    deployment_code = """
    # Example of how to load the exported model for production
    from ultralytics import YOLO

    # 1. Load the best weights
    # In production, use the absolute path to the best.pt or exported ONNX/TFLite model
    model = YOLO('runs/detect/pothole_detection/weights/best.pt')

    # 2. Define a function for prediction
    def predict_pothole(image_path):
        results = model.predict(source=image_path, conf=0.5)
        # Process results to JSON for API response
        json_results = results[0].tojson()
        return json_results

    print('Production deployment logic is ready for implementation.')
    """
    print(deployment_code)

deployment_example()

[12/15] Production Deployment Example (FastAPI/Flask Structure)

    # Example of how to load the exported model for production
    from ultralytics import YOLO
    
    # 1. Load the best weights
    # In production, use the absolute path to the best.pt or exported ONNX/TFLite model
    model = YOLO('runs/detect/pothole_detection/weights/best.pt')
    
    # 2. Define a function for prediction
    def predict_pothole(image_path):
        results = model.predict(source=image_path, conf=0.5)
        # Process results to JSON for API response
        json_results = results[0].tojson()
        return json_results
    
    print('Production deployment logic is ready for implementation.')
    


In [109]:
def deployment_example():
    print("[12/15] Production Deployment Example (FastAPI/Flask Structure)")
    deployment_code = """
    # Example of how to load the exported model for production
    from ultralytics import YOLO

    # 1. Load the best weights
    # In production, use the absolute path to the best.pt or exported ONNX/TFLite model
    model = YOLO('runs/detect/pothole_detection/weights/best.pt')

    # 2. Define a function for prediction
    def predict_pothole(image_path):
        results = model.predict(source=image_path, conf=0.5)
        # Process results to JSON for API response
        json_results = results[0].tojson()
        return json_results

    print('Production deployment logic is ready for implementation.')
    """
    print(deployment_code)

deployment_example()

[12/15] Production Deployment Example (FastAPI/Flask Structure)

    # Example of how to load the exported model for production
    from ultralytics import YOLO
    
    # 1. Load the best weights
    # In production, use the absolute path to the best.pt or exported ONNX/TFLite model
    model = YOLO('runs/detect/pothole_detection/weights/best.pt')
    
    # 2. Define a function for prediction
    def predict_pothole(image_path):
        results = model.predict(source=image_path, conf=0.5)
        # Process results to JSON for API response
        json_results = results[0].tojson()
        return json_results
    
    print('Production deployment logic is ready for implementation.')
    


In [108]:
def deployment_example():
    print("[12/15] Production Deployment Example (FastAPI/Flask Structure)")
    deployment_code = """
    # Example of how to load the exported model for production
    from ultralytics import YOLO

    # 1. Load the best weights
    # In a production environment, you would use the exported ONNX or TFLite model
    model = YOLO('runs/detect/pothole_detection/weights/best.pt')

    # 2. Define a function for API endpoint
    def predict_pothole(image_path):
        results = model.predict(source=image_path, conf=0.5)
        # Process results to JSON for API response
        json_results = results[0].tojson()
        return json_results

    print('Deployment logic ready.')
    """
    print(deployment_code)

deployment_example()

[12/15] Production Deployment Example (FastAPI/Flask Structure)

    # Example of how to load the exported model for production
    from ultralytics import YOLO
    
    # 1. Load the best weights
    # In a production environment, you would use the exported ONNX or TFLite model
    model = YOLO('runs/detect/pothole_detection/weights/best.pt')
    
    # 2. Define a function for API endpoint
    def predict_pothole(image_path):
        results = model.predict(source=image_path, conf=0.5)
        # Process results to JSON for API response
        json_results = results[0].tojson()
        return json_results
    
    print('Deployment logic ready.')
    


In [107]:
def deployment_example():
    print("[12/15] Production Deployment Example (FastAPI/Flask Structure)")
    deployment_code = """
    # Example of how to load the exported model for production
    from ultralytics import YOLO

    # 1. Load the best weights
    # In a production environment, you would use the exported ONNX or TFLite model
    model = YOLO('runs/detect/pothole_detection/weights/best.pt')

    # 2. Define a function for API endpoint
    def predict_pothole(image_path):
        results = model.predict(source=image_path, conf=0.5)
        # Process results to JSON for API response
        json_results = results[0].tojson()
        return json_results

    print('Deployment logic ready.')
    """
    print(deployment_code)

deployment_example()

[12/15] Production Deployment Example (FastAPI/Flask Structure)

    # Example of how to load the exported model for production
    from ultralytics import YOLO
    
    # 1. Load the best weights
    # In a production environment, you would use the exported ONNX or TFLite model
    model = YOLO('runs/detect/pothole_detection/weights/best.pt')
    
    # 2. Define a function for API endpoint
    def predict_pothole(image_path):
        results = model.predict(source=image_path, conf=0.5)
        # Process results to JSON for API response
        json_results = results[0].tojson()
        return json_results
    
    print('Deployment logic ready.')
    


In [106]:
def deployment_example():
    print("[12/15] Production Deployment Example (FastAPI/Flask Structure)")
    deployment_code = """
    # Example of how to load the exported model for production
    from ultralytics import YOLO

    # 1. Load the best weights
    # In a production environment, you would use the exported ONNX or TFLite model
    model = YOLO('runs/detect/pothole_detection/weights/best.pt')

    # 2. Define a function for API endpoint
    def predict_pothole(image_path):
        results = model.predict(source=image_path, conf=0.5)
        # Process results to JSON for API response
        json_results = results[0].tojson()
        return json_results

    print('Deployment logic ready.')
    """
    print(deployment_code)

deployment_example()

[12/15] Production Deployment Example (FastAPI/Flask Structure)

    # Example of how to load the exported model for production
    from ultralytics import YOLO
    
    # 1. Load the best weights
    # In a production environment, you would use the exported ONNX or TFLite model
    model = YOLO('runs/detect/pothole_detection/weights/best.pt')
    
    # 2. Define a function for API endpoint
    def predict_pothole(image_path):
        results = model.predict(source=image_path, conf=0.5)
        # Process results to JSON for API response
        json_results = results[0].tojson()
        return json_results
    
    print('Deployment logic ready.')
    


In [105]:
def deployment_example():
    print("[12/15] Production Deployment Example (FastAPI/Flask Structure)")
    deployment_code = """
    # Example of how to load the exported model for production
    from ultralytics import YOLO

    # 1. Load the best weights
    model = YOLO('runs/detect/pothole_detection/weights/best.pt')

    # 2. Define a function for API endpoint
    def predict_pothole(image_path):
        results = model.predict(source=image_path, conf=0.5)
        # Process results to JSON for API response
        json_results = results[0].tojson()
        return json_results

    print('Deployment logic ready.')
    """
    print(deployment_code)

deployment_example()

[12/15] Production Deployment Example (FastAPI/Flask Structure)

    # Example of how to load the exported model for production
    from ultralytics import YOLO
    
    # 1. Load the best weights
    model = YOLO('runs/detect/pothole_detection/weights/best.pt')
    
    # 2. Define a function for API endpoint
    def predict_pothole(image_path):
        results = model.predict(source=image_path, conf=0.5)
        # Process results to JSON for API response
        json_results = results[0].tojson()
        return json_results
    
    print('Deployment logic ready.')
    


In [97]:
def evaluate_model(model):
    print("[8/15] Evaluating Model Performance...")
    try:
        metrics = model.val()
        print(f"mAP@50: {metrics.box.map50:.4f}")
        print(f"mAP@50-95: {metrics.box.map:.4f}")
        print(f"Precision: {metrics.box.mp:.4f}")
        print(f"Recall: {metrics.box.mr:.4f}")
        return metrics
    except Exception as e:
        print(f"Evaluation failed: {e}. Ensure training completed successfully.")

evaluate_model(model)

[8/15] Evaluating Model Performance...
Evaluation failed: 'NoneType' object has no attribute 'val'. Ensure training completed successfully.


In [98]:
def visualize_results():
    print("[9/15] Generating Training Visualizations...")
    results_dir = Path('/content/runs/detect/pothole_detection')

    if results_dir.exists():
        if (results_dir / 'results.png').exists():
            plt.figure(figsize=(15, 10))
            img = mpimg.imread(str(results_dir / 'results.png'))
            plt.imshow(img)
            plt.axis('off')
            plt.title("Training Metrics and Loss Curves")
            plt.show()

        if (results_dir / 'confusion_matrix.png').exists():
            plt.figure(figsize=(10, 10))
            img = mpimg.imread(str(results_dir / 'confusion_matrix.png'))
            plt.imshow(img)
            plt.axis('off')
            plt.title("Confusion Matrix")
            plt.show()
    else:
        print("Results directory not found. Visualizations skipped.")

visualize_results()

[9/15] Generating Training Visualizations...


In [99]:
def export_model(model):
    print("[10/15] Exporting Model to ONNX and TFLite...")
    try:
        onnx_path = model.export(format='onnx')
        tflite_path = model.export(format='tflite')
        print(f"Model exported to: {onnx_path} and {tflite_path}")
    except Exception as e:
        print(f"Export failed: {e}")

export_model(model)

[10/15] Exporting Model to ONNX and TFLite...
Export failed: 'NoneType' object has no attribute 'export'


In [100]:
def run_custom_inference():
    print("[11/15] Custom Inference Module...")
    test_images = list(Path('/content/dataset/test/images').glob('*.jpg'))
    if test_images:
        results = model.predict(source=str(test_images[0]), save=True, conf=0.25)
        print(f"Inference complete. Results saved to {results[0].save_dir}")
    else:
        print("No test images found for sample inference.")

run_custom_inference()

[11/15] Custom Inference Module...
No test images found for sample inference.


In [93]:
def evaluate_model(model):
    print("[8/15] Evaluating Model Performance...")
    try:
        metrics = model.val()
        print(f"mAP@50: {metrics.box.map50:.4f}")
        print(f"mAP@50-95: {metrics.box.map:.4f}")
        print(f"Precision: {metrics.box.mp:.4f}")
        print(f"Recall: {metrics.box.mr:.4f}")
        return metrics
    except Exception as e:
        print(f"Evaluation failed: {e}. Ensure training completed successfully.")

evaluate_model(model)

[8/15] Evaluating Model Performance...
Evaluation failed: 'NoneType' object has no attribute 'val'. Ensure training completed successfully.


In [94]:
def visualize_results():
    print("[9/15] Generating Training Visualizations...")
    results_dir = Path('/content/runs/detect/pothole_detection')

    if results_dir.exists():
        # Plot results.png (Loss and Metrics)
        if (results_dir / 'results.png').exists():
            plt.figure(figsize=(15, 10))
            img = mpimg.imread(str(results_dir / 'results.png'))
            plt.imshow(img)
            plt.axis('off')
            plt.title("Training Metrics and Loss Curves")
            plt.show()

        # Plot Confusion Matrix
        if (results_dir / 'confusion_matrix.png').exists():
            plt.figure(figsize=(10, 10))
            img = mpimg.imread(str(results_dir / 'confusion_matrix.png'))
            plt.imshow(img)
            plt.axis('off')
            plt.title("Confusion Matrix")
            plt.show()
    else:
        print("Results directory not found. Visualizations skipped.")

visualize_results()

[9/15] Generating Training Visualizations...


In [95]:
def export_model(model):
    print("[10/15] Exporting Model to ONNX and TFLite...")
    try:
        onnx_path = model.export(format='onnx')
        tflite_path = model.export(format='tflite')
        print(f"Model exported to: {onnx_path} and {tflite_path}")
    except Exception as e:
        print(f"Export failed: {e}")

export_model(model)

[10/15] Exporting Model to ONNX and TFLite...
Export failed: 'NoneType' object has no attribute 'export'


In [96]:
def run_custom_inference():
    print("[11/15] Custom Inference Module...")
    test_images = list(Path('/content/dataset/test/images').glob('*.jpg'))
    if test_images:
        results = model.predict(source=str(test_images[0]), save=True, conf=0.25)
        print(f"Inference complete. Results saved to {results[0].save_dir}")
    else:
        print("No test images found for sample inference.")

run_custom_inference()

[11/15] Custom Inference Module...
No test images found for sample inference.


In [89]:
def evaluate_model(model):
    print("[8/15] Evaluating Model Performance...")
    try:
        metrics = model.val()
        print(f"mAP@50: {metrics.box.map50:.4f}")
        print(f"mAP@50-95: {metrics.box.map:.4f}")
        print(f"Precision: {metrics.box.mp:.4f}")
        print(f"Recall: {metrics.box.mr:.4f}")
        return metrics
    except Exception as e:
        print(f"Evaluation failed: {e}. Ensure training completed successfully.")

evaluate_model(model)

[8/15] Evaluating Model Performance...
Evaluation failed: 'NoneType' object has no attribute 'val'. Ensure training completed successfully.


In [90]:
def visualize_results():
    print("[9/15] Generating Training Visualizations...")
    results_dir = Path('/content/runs/detect/pothole_detection')

    if results_dir.exists():
        # Plot results.png (Loss and Metrics)
        if (results_dir / 'results.png').exists():
            plt.figure(figsize=(15, 10))
            img = mpimg.imread(str(results_dir / 'results.png'))
            plt.imshow(img)
            plt.axis('off')
            plt.title("Training Metrics and Loss Curves")
            plt.show()

        # Plot Confusion Matrix
        if (results_dir / 'confusion_matrix.png').exists():
            plt.figure(figsize=(10, 10))
            img = mpimg.imread(str(results_dir / 'confusion_matrix.png'))
            plt.imshow(img)
            plt.axis('off')
            plt.title("Confusion Matrix")
            plt.show()
    else:
        print("Results directory not found. Visualizations skipped.")

visualize_results()

[9/15] Generating Training Visualizations...


In [91]:
def export_model(model):
    print("[10/15] Exporting Model to ONNX and TFLite...")
    try:
        onnx_path = model.export(format='onnx')
        tflite_path = model.export(format='tflite')
        print(f"Model exported to: {onnx_path} and {tflite_path}")
    except Exception as e:
        print(f"Export failed: {e}")

export_model(model)

[10/15] Exporting Model to ONNX and TFLite...
Export failed: 'NoneType' object has no attribute 'export'


In [92]:
def run_custom_inference():
    print("[11/15] Custom Inference Module...")
    # Example: Run inference on a test image if exists
    test_images = list(Path('/content/dataset/test/images').glob('*.jpg'))
    if test_images:
        results = model.predict(source=str(test_images[0]), save=True, conf=0.25)
        print(f"Inference complete. Results saved to {results[0].save_dir}")
    else:
        print("No test images found for sample inference.")

run_custom_inference()

[11/15] Custom Inference Module...
No test images found for sample inference.


In [82]:
def setup_dataset():
    print("[3/15] Downloading Pothole Dataset...")
    import os
    import zipfile
    import shutil
    import subprocess

    # Using a reliable direct link to a YOLO-formatted pothole dataset
    url = "https://github.com/nandinibagga/pothole-detection/raw/master/dataset.zip"

    try:
        if os.path.exists('/content/dataset'):
            shutil.rmtree('/content/dataset')
        os.makedirs('/content/dataset', exist_ok=True)

        print("Downloading dataset...")
        subprocess.run(['curl', '-L', url, '-o', '/content/pothole.zip'], check=True)

        if os.path.exists('/content/pothole.zip') and zipfile.is_zipfile('/content/pothole.zip'):
            with zipfile.ZipFile('/content/pothole.zip', 'r') as zip_ref:
                zip_ref.extractall('/content/dataset')
            print("\nDataset downloaded and extracted successfully to /content/dataset")

            # Clean up: remove the zip file and list contents to verify
            os.remove('/content/pothole.zip')
            print("Contents:", os.listdir('/content/dataset'))
        else:
            print("Primary source failed. Please manually upload the dataset or check the URL.")
    except Exception as e:
        print(f"\nCritical failure during dataset setup: {e}")

setup_dataset()

[3/15] Downloading Pothole Dataset...
Primary source failed. Please manually upload the dataset or check the URL.
